<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/03-rag/02-hybrid-and-reranking.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 混合检索与重排序

**目标：** 把关键词检索和向量检索结合起来，加上一个 reranker，并度量每一环是否值回它的成本。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，接下来的两个单元格会把它从零搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的环境准备 helper，并拉入 `groq` 客户端；本篇还会用到 `requests`、`sentence-transformers`、`numpy`、`rank-bm25`。
2. **载入你的 API key。** 免费 key 在 [console.groq.com](https://console.groq.com/) 领（无需信用卡）。在 Colab 里通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称必须严格写成 `GROQ_API_KEY`，粘贴值，并把 **Notebook access** 打开。本地运行的话，则把 `GROQ_API_KEY` 设成环境变量。

（完整走查和选模型的指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" requests sentence-transformers numpy rank-bm25

In [ ]:
from aien import setup

# 载入 GROQ_API_KEY（Colab Secrets 或本地环境变量）并返回一个可直接
# 使用的 Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用
# 报 404，就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 问题：单一检索器覆盖不了真实的查询流

Notebook 01 以一个分裂的判决收尾：向量搜索稳稳拿下同义改写，却在精确标识符上失手，而一个笨笨的子串 grep 恰好相反。这不是学术上的好奇。你的生产查询流里，*两类*同时都有。有人粘贴 `Sec-WebSocket-Accept`，想看到精确的那一节；另一个人问「怎么让连接保持双向开放？」，想要的是同一篇 WebSocket RFC。没有任何单一检索器能同时服务好这两个用户。

本篇通过叠加组件来解决它，但你加进去的每个阶段都要付出延迟和金钱，所以真正的本事是知道哪些阶段值回它的位置。我们分三步搭建，从最便宜的开始：

| 阶段 | 它是什么 | 它能抓住什么 | 成本 | 结论 |
|---|---|---|---|---|
| **BM25** | 关键词打分（长大成人的 TF-IDF） | 精确标识符、稀有 token、错误字符串 | 几乎免费，无 API | 不肯退场的基线 |
| **向量搜索** | bi-encoder 余弦（来自 nb 01） | 同义改写、概念性问题 | 本地几乎免费 | 补足 BM25，但取代不了它 |
| **混合（RRF）** | 按名次融合两份排序列表 | 在*两类*查询上都稳 | 几乎免费 | 几乎全是收益，加上它 |
| **Reranker** | 把 query+passage 一起读，给相关性打分 | 前两者看不到的「提及 vs 回答」之别 | 昂贵（每查询 N 次调用） | 只有评测说它值，才留 |

贯穿始终的一条：**按最便宜优先的顺序叠加阶段，并让每一阶段在留下之前证明自己值回成本。** 最后一节度量的正是这件事。

## 环境准备：语料、chunk、embedding

语料和按 section 切出的 chunk 与 notebook 01 相同（这些单元格被重复了一遍，因为这里每个 notebook 都自包含），向量索引也以同样的方式重建。分块器是精简复制；分块本身要到 notebook 03 才拆开讲。


In [ ]:
# 下载共享语料：十篇 IETF RFC，缓存到 data/rfc/。
# 本仓库里每个需要这份语料的 notebook 都包含这个单元格 ——
# 自包含优先于 DRY，好让每个 notebook 都能自己从头跑到尾。
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc 编号 -> 原始文本
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # 已缓存：重跑时跳过下载
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

In [ ]:
# 清洗 + 按 section 切分的精简复制。notebook 03（分块）
# 才是把它逐步搭起来并解释其权衡的地方；这里只是一个既定条件。
import re

def clean_rfc(text):
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.match(r'^.*\[Page \d+\]\s*$', l)        # 页脚
             and not re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', l)]  # 页眉
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_paragraphs(text, max_chars=2400):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # 长 section：按段落打包，并把标题一起带上
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

chunk_texts, chunk_meta = [], []
for num in RFC_NUMBERS:
    for c in chunk_sections(clean_rfc(corpus[num])):
        chunk_texts.append(c)
        chunk_meta.append({'rfc': num, 'title': RFC_TITLES[num]})
print(f'{len(chunk_texts)} chunks across {len(RFC_NUMBERS)} RFCs')

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts, normalize_embeddings=True,
                             show_progress_bar=True)

def vector_search(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

## BM25：那个不肯退场的关键词基线

BM25 是数学更好的 TF-IDF：它按一个 chunk 包含多少查询词来打分，再按每个词在全语料里的常见程度和 chunk 长度打折。没有神经网络、没有训练、没有 GPU，可它至今仍是 embedding 论文必须打败的检索基线。

Notebook 01 结尾时，向量搜索在精确标识符上输给了一个子串 grep。BM25 就是那个长大成人的 grep。


In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r'[a-z0-9][a-z0-9._-]*', text.lower())

bm25 = BM25Okapi([tokenize(t) for t in chunk_texts])

def bm25_search(query, k=5):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

def show(results, label):
    print(f'--- {label}')
    for score, i in results:
        first_line = chunk_texts[i].strip().split('\n')[0][:70]
        print(f'  {score:7.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})  {first_line}')
    print()

## 各自在哪里赢

两个查询，两种检索器都用上。先是一个精确标识符 `RST`，TCP 的重置标志。然后是一句同义改写，和回答它的文本没有任何关键词重叠。


In [ ]:
q1 = 'when does TCP send a RST segment?'
show(bm25_search(q1), f'BM25:   {q1}')
show(vector_search(q1), f'vector: {q1}')

q2 = 'how can a browser keep a two-way conversation open with a server?'
show(bm25_search(q2), f'BM25:   {q2}')
show(vector_search(q2), f'vector: {q2}')

跑一下，注意那个分裂的判决：

- 在 **RST 查询**上，BM25 稳稳命中：`rst` 是稀有 token，包含它的 chunk 会占据优势。向量搜索捞回的是总体上带 TCP 味道的文本，而真正讲 RST 处理的那几节能不能挤进前 5，则未必——因为一个三个字母的标识符在 384 维的 embedding 里几乎留不下痕迹。
- 在 **双向对话查询**上，BM25 几乎没什么可抓（WebSocket RFC 说的是 "bidirectional" 和 "full-duplex"，不是 "two-way conversation"），而向量搜索轻松落到 RFC 6455 上，因为哪怕用词不同，*含义*是对上的。

两者都没有在两个查询上通吃。你的生产查询流里两类都有：有人粘贴错误字符串和标识符，也有人问含糊的概念性问题。所以你两个检索器都想要，这就引出了如何合并两份排序列表的问题。

## 混合：倒数排名融合

你不能对分数取平均：BM25 分数无上界且依赖语料，余弦分数落在 [-1, 1]。它们不在可比的尺度上，任何加权求和都是一个带旋钮的谎言。**倒数排名融合（RRF）** 绕开了这个问题：丢掉分数，只用*名次*——每份列表对每篇文档贡献 `1 / (k + rank)`，然后求和。任一列表里第 1 名很值钱，第 40 名几乎一文不值，而 `k=60`（标准常数）会削平第 1 名和第 3 名之间的差距，让一份列表无法独大。


In [ ]:
def rrf(rankings, k=60, top_n=20):
    """rankings：由 chunk 索引构成的多份已排序列表。返回 [(idx, score), ...]。"""
    scores = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])[:top_n]

def hybrid_search(query, top_n=20, pool=50):
    v = [i for _, i in vector_search(query, pool)]
    b = [i for _, i in bm25_search(query, pool)]
    return rrf([v, b], top_n=top_n)

for q in [q1, q2]:
    fused = hybrid_search(q, top_n=5)
    show([(s, i) for i, s in fused], f'hybrid: {q}')

跑一下，注意混合检索在*两个*查询上都站住了：RST 那几节活了下来，因为 BM25 把它们排在第一；WebSocket 那几节活了下来，因为向量搜索做到了。混合检索买到的是跨查询类型的这种稳健性，而不是在某个单一查询上更高的上限。

## 重排序：第二个更贵的意见

两种检索器都是*各自独立地、廉价地*给每个 chunk 针对查询打分。BM25 从没读过句子，bi-encoder 则先把查询和 chunk 压成向量再比较。**reranker** 把查询和段落*一起*读，给真实的相关性打分。这准确得多，也贵得多，所以它只在前 20 个融合候选上跑，而不是整个语料。

我们用模型本身做一个 pointwise reranker：每段一次便宜调用，「给相关性打 0–10 分，只回数字」。要清楚这是什么：**每个查询 N 次 API 调用。** 生产系统用 cross-encoder reranker（Cohere Rerank、某个托管 API，或本地模型），它以一小部分成本和延迟做同样的「一起读」把戏。概念完全一样，所以这样学到的能直接迁移过去。


In [ ]:
def llm_score(query, passage):
    prompt = (
        'Rate how relevant this passage is for answering the query, '
        'on a scale of 0 (irrelevant) to 10 (directly answers it).\n\n'
        f'Query: {query}\n\nPassage:\n{passage[:1500]}\n\n'
        'Reply with a single integer 0-10 and nothing else.'
    )
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=8,
        messages=[{'role': 'user', 'content': prompt}],
    )
    text = resp.choices[0].message.content
    m = re.search(r'\d+', text)
    return min(int(m.group()), 10) if m else 0

def rerank(query, candidate_ids):
    scored = [(llm_score(query, chunk_texts[i]), i) for i in candidate_ids]
    return sorted(scored, key=lambda x: -x[0])

In [ ]:
query = 'when does TCP send a RST segment?'

fused = hybrid_search(query, top_n=20)
candidates = [i for i, _ in fused]
reranked = rerank(query, candidates)   # 20 次 API 调用 —— 盯着墙钟时间看

print('hybrid top-5 (by RRF)          vs   reranked top-5 (by LLM 0-10)')
for (i_h, s_h), (s_r, i_r) in zip(fused[:5], reranked[:5]):
    left = chunk_texts[i_h].strip().split('\n')[0][:32]
    right = chunk_texts[i_r].strip().split('\n')[0][:32]
    print(f'  {s_h:.4f} {left:<34} {s_r:>2}/10 {right}')

跑一下，对比这两列。在这份语料上的典型结果：reranker 把真正*定义* RST 行为的那一节，提拔到只是*频繁提及* RST 的那些节之前。这正是词频统计和向量邻近性看不到的「提及 vs 回答」之别。有时候两份列表差别很小；那也是数据（见下文）。

## 诚实的账：每个阶段都必须值回它的成本

检索流水线会不断累积阶段，因为每个阶段听起来都挺合理。真正的纪律是：针对*你自己的*查询，度量每个阶段在结果质量上增加了什么，对照它在延迟和金钱上花了什么。


In [ ]:
import time

query = 'how does a server tell a client to retry later?'

t0 = time.perf_counter()
v = vector_search(query, 50)
t1 = time.perf_counter()
b = bm25_search(query, 50)
t2 = time.perf_counter()
fused = rrf([[i for _, i in v], [i for _, i in b]], top_n=20)
t3 = time.perf_counter()
reranked = rerank(query, [i for i, _ in fused])
t4 = time.perf_counter()

print(f'vector search : {(t1 - t0) * 1000:8.1f} ms')
print(f'bm25 search   : {(t2 - t1) * 1000:8.1f} ms')
print(f'rrf fusion    : {(t3 - t2) * 1000:8.1f} ms')
print(f'LLM rerank    : {(t4 - t3) * 1000:8.1f} ms   (20 API calls)')
print()
show([(s, i) for i, s in fused[:5]], 'hybrid top-5')
show(reranked[:5], 'reranked top-5')

跑一下，研究这些数字的形状，而不是精确取值：

- **BM25 和融合实际上免费：** 微秒到低毫秒，没有 API 成本。把 BM25 加进向量流水线几乎全是收益。它不花任何可度量的成本，却救回了整整一类精确标识符查询。它轻易值回成本。
- **LLM 重排序是另一回事：** 串行几十秒（20 次往返），每个查询大约 1 万输入 token，在 Groq 上免费，但按典型付费 API 定价就是几分钱。生产用的 cross-encoder 会把它压到几十毫秒和零点几分钱，但即便如此，它仍是每个候选最贵的阶段。只有最终答案变好了，它才值回成本，而这取决于你那融合后的前 5 名有多经常错——且错法是 reranker 能修的那种。

而最后那句正是陷阱：你没法靠肉眼扫两个查询就把它定下来。重排序是否值回它的位置是一个*评测*问题。你需要一组已知答案的问题，以及一个命中率，来对比有它和没它。Notebook 04 手工搭一个小规模的；第 04 节把它做严谨。在你能度量之前，忍住别加阶段。


## 最佳实践与反模式

| ✅ 要做的 | ❌ 反模式 |
|---|---|
| 在向量之外并上 BM25；它几乎免费，还能救回精确标识符查询 | 只上向量，丢掉每一次缩写/错误码查找 |
| 用 RRF（基于名次）融合排序列表 | 对 BM25 和余弦分数取平均（尺度不同，一个带旋钮的谎言） |
| 只有当评测表明它值回延迟/成本时，才加 reranker | 因为每个阶段「听起来都合理」就一路叠 reranker |
| 只重排一个小候选池（前 20 个），而不是整个语料 | 让昂贵的 cross-encoder 跑遍所有东西 |
| 用命中率而不是两个查询来定夺「这一阶段有帮助吗？」 | 肉眼扫两个例子就宣布胜利 |

（整套技术栈范围的清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习

1. **加权 RRF。** 给 `rrf()` 加一个按列表的权重（把每份列表的贡献乘上权重）。在 `q1` 和 `q2` 上试试把向量权重设为 BM25 的 2 倍，再反过来。找一个加权会翻转头名结果的查询，然后判断：没有评测的话，你能向同事把这个旋钮解释得通吗？
2. **把 reranker 批量化。** 改写 `rerank()`，用*单次* API 调用给全部 20 个段落打分：在一个 prompt 里给段落编号，要求返回一个「段落编号 → 分数」的 JSON 对象。度量延迟和成本的下降，再检查它的分数有多少时候和 pointwise 版本不一致。你正在重新发现 listwise 与 pointwise 的取舍。
3. **重排序深度扫描。** 对你自己选的三个查询，分别重排融合候选的前 5、10、20 个。到哪个深度时最终的前 3 不再变化？那个深度就是你真实的候选预算；超过它的每一分钱都是没有效果的支出。
4. **廉价模型 reranker。** 把 reranker 换成 `openai/gpt-oss-20b`，在同样这 20 个候选上比较它的 0–10 分与 120B 的。如果它们给出同样的*排名*，更便宜的模型就该干这份活——这是你度量之后把某个流水线阶段降配的许多次里的第一次。